In [ ]:

from datasets import Dataset, load_from_disk, DatasetDict, Features, Sequence, Value
import pandas as pd
import json
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    DataCollatorWithPadding,
    EvalPrediction,
)
import evaluate as hf_evaluate
from trainer_qa import QuestionAnsweringTrainer
from utils_qa import postprocess_qa_predictions
from retrieval import SparseRetrieval, DenseRetrieval, BM25Retrieval
from arguments import (
    # ModelArguments,
    # DataTrainingArguments,
    RetrievalArguments,
)
import logging

logger = logging.getLogger(__name__)



: 

In [6]:
import sys
print(sys.executable)

/data/ephemeral/home/.py310/bin/python


## 1. Load Train, Validation Dataset

In [7]:

# Train, Validation
dataset = load_from_disk("../data/train_dataset/")

train = dataset['train']
val = dataset['validation']

## 2. Load Model, Tokenizer

In [31]:
model = AutoModelForQuestionAnswering.from_pretrained("klue/roberta-large")

/data/ephemeral/home/.py310/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Some weights of RobertaForQuestionAnswering were not initialized from the model checkpoint at klue/roberta-large and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [33]:
tokenizer = AutoTokenizer.from_pretrained("klue/roberta-large")

## 3. Set arguments(Trainer, Retrieval)

In [34]:
training_args = TrainingArguments(
    output_dir="./outputs",
    do_train=False,
    do_eval=False,
    do_predict=True,
    per_device_eval_batch_size=4,
)

In [35]:
retrieval_args = RetrievalArguments(
    data_path = "../data",
    context_path = "wikipedia_documents.json",
    top_k = 10,
    use_faiss = False,
    num_clusters = 64,
    retrieval_type = "bm25",
    dense_model_name_or_path = None,
)

## 4. Make train features

In [ ]:
def prepare_train_features(examples):
    tokenized_examples = tokenizer(
      examples['question'],
      examples['context'],
      truncation="only_second",
      max_length=512,
      stride=128,
      return_overflowing_tokens=True,
      return_offsets_mapping=True,
      return_token_type_ids=False,
      padding="max_length"
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized_examples.sequence_ids(i)

        sample_index = sample_mapping[i]
        answers = examples['answers'][sample_index]

        if len(answers["answer_start"]) == 0:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            # [수정 포인트 2] Context(1) 시작 위치 찾기 (안전장치 추가)
            token_start_index = 0
            while token_start_index < len(sequence_ids) and sequence_ids[token_start_index] != 1:
                token_start_index += 1

            token_end_index = len(input_ids) - 1
            while token_end_index >= 0 and sequence_ids[token_end_index] != 1:
                token_end_index -= 1

            # Context가 없는 경우 (매우 드물지만 안전하게 처리)
            if token_start_index >= len(sequence_ids) or token_end_index < 0:
                tokenized_examples["start_positions"].append(cls_index)
                tokenized_examples["end_positions"].append(cls_index)
                continue

            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                tokenized_examples["start_positions"].append(cls_index)
                tokenized_examples["end_positions"].append(cls_index)
            else:
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                tokenized_examples["start_positions"].append(token_start_index - 1)
                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized_examples["end_positions"].append(token_end_index + 1)

    # 4. [수정 포인트 3] 사용자님이 작성하신 디버깅 코드 (위치 및 들여쓰기 수정)
    vocab_size = tokenizer.vocab_size
    for idx, input_ids in enumerate(tokenized_examples["input_ids"]):
        max_token_id = max(input_ids)
        if max_token_id >= vocab_size:
            logger.error(f"🚨 [Error Found] Sample Index: {idx}")
            logger.error(f"   Max Token ID: {max_token_id} (Vocab Size: {vocab_size})")
            logger.error(f"   Input IDs: {input_ids}")
            raise ValueError(f"토큰 ID({max_token_id})가 Vocab Size({vocab_size})를 초과했습니다! 캐시 문제일 가능성이 높습니다.")

    return tokenized_examples

: 

In [37]:
train_datasets = train.map(
  prepare_train_features,
  batched=True,
  remove_columns=train.column_names,
)

Map:   0%|          | 0/3952 [00:00<?, ? examples/s]

In [38]:
train_datasets

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 5769
})

## 5. Make validation features

In [39]:
def prepare_validation_features(examples):
  tokenized_examples = tokenizer(
      examples['question'],
      examples['context'],
      truncation="only_second",
      max_length=512,
      stride=128,
      return_overflowing_tokens=True,
      return_offsets_mapping=True,
      padding="max_length"
  )

  sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
  tokenized_examples["example_id"] = []

  for i in range(len(tokenized_examples["input_ids"])):
      sequence_ids = tokenized_examples.sequence_ids(i)
      context_index = 1

      sample_index = sample_mapping[i]
      tokenized_examples["example_id"].append(
          examples["id"][sample_index]
      )

      tokenized_examples["offset_mapping"][i] = [
          (o if sequence_ids[k] == context_index else None)
          for k, o in enumerate(tokenized_examples["offset_mapping"][i])]

  return tokenized_examples


In [40]:
val_datasets =val.map(
  prepare_validation_features,
  batched=True,
  remove_columns=val.column_names,
)

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

In [41]:
val_datasets

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 351
})

## 6. Create_trainer Method

In [42]:
def create_trainer(train_dataset, eval_dataset, eval_examples, training_args) -> QuestionAnsweringTrainer:
  data_collator = DataCollatorWithPadding(
    tokenizer,
    pad_to_multiple_of=8,
  )
  squad_metric = hf_evaluate.load("squad")

  def post_processing_function(examples, features, predictions, training_args):
    return postprocess_qa_predictions(
      examples=examples,
      features=features,
      predictions=predictions,
      max_answer_length=30,
      output_dir=training_args.output_dir,
    )
  def compute_metrics(p) :

    # 1. 예측값(Prediction) 포맷팅
    formatted_predictions = [
        {"id": k, "prediction_text": v} for k, v in p.items()
    ]

    # 2. 정답지(Reference) 가져오기
    # eval_examples는 create_trainer의 인자로 들어온 데이터셋(Validation)입니다.
    references = [
        {"id": ex["id"], "answers": ex["answers"]} for ex in eval_examples
    ]

    return squad_metric.compute(predictions=formatted_predictions, references=references)

  trainer = QuestionAnsweringTrainer(
    model,
    training_args,
    train_dataset,
    eval_dataset,
    eval_examples,
    tokenizer,
    data_collator,
    post_process_function=post_processing_function,
    compute_metrics=compute_metrics,
  )
  return trainer

## 7. Make Retriever (TF-IDF, BM25, Dense ...)

In [43]:
if retrieval_args.retrieval_type == 'dense':
  retriever = DenseRetrieval(
    model_name_or_path=retrieval_args.dense_model_name_or_path,
    data_path=retrieval_args.data_path,
    context_path=retrieval_args.context_path,
  )
elif retrieval_args.retrieval_type == 'sparse':
  retriever = SparseRetrieval(
    tokenize_fn=tokenizer.tokenize,
    data_path=retrieval_args.data_path,
    context_path=retrieval_args.context_path,
  )
else: #BM25, Hybrid ... 
    retriever = BM25Retrieval(
        tokenize_fn=tokenizer.tokenize,
        data_path=retrieval_args.data_path,
        context_path=retrieval_args.context_path,
    )

Lengths of unique contexts : 56737


In [44]:
print(retriever)

# 8. Return top-k Doc. Method

In [45]:
def retrieve_for_odqa(queries):
  """
    ODQA 검색을 수행합니다.
    1. 검색을 위한 인덱스(임베딩)가 없으면 생성합니다.
    2. 질문 형태(단일/데이터셋)에 맞춰 검색 결과를 반환합니다.
  """

  # retriever.get_sparse_embedding()
  retriever.build_index()

  # faiss 사용시

  # if retrieval_args.use_faiss:
  #       retriever.build_faiss(num_clusters=retrieval_args.num_clusters)
  #       return retriever.retrieve_faiss(queries, topk=retrieval_args.top_k)
  
  return retriever.retrieve(queries, topk=retrieval_args.top_k)

## 9. Inference

In [46]:
def answer(question):

  # 1) 문서 후보 가져오기 
  doc_scores, passages = retrieve_for_odqa(question)
  
  # 2) id/Q/context 형태로 변환
  qa_dataset = Dataset.from_dict(
    {
      "id": [f"q-0-{i}" for i in range(len(passages))],
                "question": [question] * len(passages),
                "context": passages, 
    }
  )

  # 3) MRC 데이터 전처리 & Trainer 생성

  eval_dataset = qa_dataset.map( #qa_dataset = eval_examples
  prepare_validation_features,
  batched=True,
  remove_columns=qa_dataset.column_names,
)

  trainer = create_trainer(
    train_dataset=None,
    eval_dataset=eval_dataset,
    eval_examples=qa_dataset,
    training_args=training_args,
  )

  # 4) 예측 수행

  predictions = trainer.predict(
    test_dataset=eval_dataset,
    test_examples=qa_dataset,
  )

  # 5) return format
  return {
    "question": question,
    "answers": predictions,
    "passages": passages,
    "scores": doc_scores,   
  } 


## 10. Train MRC / Evaluate ODQA

In [47]:
def train_mrc():
  
  trainer = create_trainer(
    train_dataset=train_datasets,
    eval_dataset=val_datasets,
    eval_examples=val,
    training_args=training_args,
    )
  
  # Training
  train_trainer = create_trainer(
    train_dataset=train_datasets,
    eval_dataset=None,
    eval_examples=None,
    training_args=training_args,
  )

  train_result = train_trainer.train()
  train_trainer.save_model()

  metrics = train_result.metrics
  metrics['train_samples'] = len(train_datasets)

  train_trainer.save_metrics('train', metrics)
  train_trainer.save_state()

  with open('./outputs/train_results.txt', 'w', encoding='utf-8') as writer:

    logger.info("==========  TRAIN RESULTS  ==========")
    for k, v in sorted(metrics.items()):
      logger.info(" %s = %s", k, v)
      writer.write(f'{k} = {v}\n')

  # Evaluation
  logger.info("==========  EVALUATE MRC  ==========")
  metrics = trainer.evaluate()
  metrics['eval_samples'] = len(val_datasets)

  trainer.save_metrics('eval', metrics)
  

## 11. Evaluate ODQA

In [48]:
def evaluate_odqa(pure_mrc=False):
  if pure_mrc:
    # Pure MRC Eval.
    logger.info("==========  Retrieval 평가 대상 아님. MRC만 평가합니다.  ==========")

    trainer = create_trainer(
      train_dataset = None,
      eval_dataset=val_datasets,
      eval_examples=val,
      training_args=training_args,
    )

    logger.info("==========  Evaluate MRC-only  ==========")
    metrics = trainer.evaluate()
    metrics['eval_samples'] = len(val_datasets)
    trainer.save_metrics('eval', metrics)
    
    return metrics
    
  logger.info("========  Retreiver for ODQA Evaluation  ========")

  # 1) Retrieval 수행
  df = retrieve_for_odqa(val)
  df = df[["id", "question", "context", "answers"]] # original_context 제거

  # 2) DataFrame -> Dataset

  features = Features(
    {
      "answers": Sequence(
        feature={
            "text": Value(dtype="string", id=None),
            "answer_start": Value(dtype="int32", id=None),
        },
        length=-1,
        id=None,
      ),
      "context": Value(dtype="string", id=None),
      "id": Value(dtype="string", id=None),
      "question": Value(dtype="string", id=None),
    }
  )

  odqa_dataset = Dataset.from_pandas(df, features=features)


  # 3) MRC 전처리 및 평가

  eval_dataset = odqa_dataset.map(
    prepare_validation_features,
    batched=True,
    remove_columns=odqa_dataset.column_names,
  )

  trainer = create_trainer(
    train_dataset=None,
    eval_dataset=eval_dataset,
    eval_examples=odqa_dataset,
    training_args=training_args,
  )

  logger.info("==========  EVALUATE (ODQA)  ==========")
  metrics = trainer.evaluate()
  metrics['eval_samples'] = len(eval_dataset)
  trainer.save_metrics('test', metrics)

  return metrics



## 12. Example


In [ ]:
# ============================================
# 예시 1: 단일 질문에 대한 답변 생성
# ============================================
result = answer("대한민국의 수도는 어디인가요?")
print(f"질문: {result['question']}")
print(f"답변: {result['answers']}")
print(f"검색된 문서 수: {len(result['passages'])}")


Tokenizing all contexts for BM25...


Tokenizing:   0%|          | 0/56737 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1131 > 512). Running this sequence through the model will result in indexing errors


Build BM25 index...
BM25 index build complete.


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

/data/ephemeral/home/.py310/lib/python3.10/site-packages/accelerate/accelerator.py:446: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/data/ephemeral/home/.py310/lib/python3.10/site-packages/transformers/trainer_pt_utils.py:435: FutureWarning: DistributedTensorGatherer is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


: 

In [27]:
# train_mrc() 호출 전에 확인해 보세요.

# 1. train_datasets가 None은 아닌지?
print(f"Dataset Type: {type(train_datasets)}")

# 2. 첫 번째 데이터가 정상적으로 출력되는지? (input_ids가 있어야 함)
if train_datasets is not None:
    print("First Example:", train_datasets[0])
    print("Columns:", train_datasets.column_names)
else:
    print("❌ train_datasets가 None입니다! prepare_mrc_datasets의 반환값을 확인하세요.")

Dataset Type: <class 'datasets.arrow_dataset.Dataset'>
First Example: {'input_ids': [2, 3698, 2069, 3954, 2470, 3666, 2079, 7895, 8586, 2207, 2069, 554, 2259, 3728, 3860, 2073, 35, 3, 3666, 10346, 2252, 4013, 3666, 10450, 12, 29963, 30605, 2041, 19148, 2012, 9230, 13, 1497, 1402, 2252, 2021, 2179, 3666, 4570, 2079, 10450, 28674, 18, 1, 81, 1, 81, 2044, 2226, 17352, 2052, 10450, 2079, 27345, 3622, 18, 544, 12881, 22, 2211, 2079, 10450, 5069, 2052, 6940, 2496, 2051, 3911, 2211, 2079, 10450, 5069, 6233, 3896, 2496, 2051, 1513, 2062, 18, 6724, 2259, 26, 2440, 2052, 2307, 16, 22, 2440, 10598, 3956, 2019, 2223, 1570, 21, 19, 23, 3292, 10450, 5069, 2069, 3755, 6940, 7488, 7145, 2170, 14352, 18, 1, 81, 1, 81, 2044, 2226, 10450, 2073, 3666, 11119, 2145, 2259, 4405, 2318, 3666, 3698, 2069, 12104, 6233, 1889, 2259, 3666, 7145, 7895, 2170, 4424, 5187, 2138, 1889, 2259, 3860, 28674, 18, 11119, 2052, 5387, 2145, 3674, 2170, 3618, 5851, 16, 3698, 2069, 3954, 2470, 8199, 2079, 4668, 2069, 25154, 2085,

In [28]:
# ============================================
# 예시 2: MRC 학습 및 평가
# ============================================
train_mrc()

/data/ephemeral/home/.py310/lib/python3.10/site-packages/accelerate/accelerator.py:446: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my 

Step,Training Loss
500,1.778900
1000,0.827600
1500,0.437900
2000,0.152000


/data/ephemeral/home/.py310/lib/python3.10/site-packages/transformers/trainer_pt_utils.py:435: FutureWarning: DistributedTensorGatherer is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


  0%|          | 0/240 [00:00<?, ?it/s]

In [29]:
# ============================================
# 예시 3: ODQA 평가 (Retrieval + MRC)
# ============================================
metrics = evaluate_odqa(pure_mrc=False)
print(f"ODQA 평가 결과: {metrics}")


Tokenizing all contexts for BM25...


Tokenizing:   0%|          | 0/56737 [00:00<?, ?it/s]

Build BM25 index...
BM25 index build complete.


BM25 retrieval:   0%|          | 0/240 [00:00<?, ?it/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

/data/ephemeral/home/.py310/lib/python3.10/site-packages/accelerate/accelerator.py:446: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/data/ephemeral/home/.py310/lib/python3.10/site-packages/transformers/trainer_pt_utils.py:435: FutureWarning: DistributedTensorGatherer is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


  0%|          | 0/240 [00:00<?, ?it/s]

ODQA 평가 결과: {'exact_match': 45.416666666666664, 'f1': 52.908549783549795, 'eval_samples': 3508}


In [30]:

# ============================================
# 예시 4: 순수 MRC 평가 (Retrieval 없음)
# ============================================
metrics = evaluate_odqa(pure_mrc=True)
print(f"MRC 평가 결과: {metrics}")

Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


  0%|          | 0/240 [00:00<?, ?it/s]

MRC 평가 결과: {'exact_match': 56.25, 'f1': 65.9050925925926, 'eval_samples': 351}


## ✔ Dataset 분석을 위한 DataFrame 변환

In [ ]:

# Wiki (Retrieval Data)
with open("../data/wikipedia_documents.json", 'r', encoding='utf-8') as f:
    wiki_data = json.load(f)

wiki_data_list = list(wiki_data.values())
wiki_dataset = Dataset.from_list(wiki_data_list)


In [ ]:

# Test
test_dataset = load_from_disk("../data/test_dataset/")
test = test_dataset['validation']

In [ ]:

# Dataset -> DataFrame
train_df = pd.DataFrame(train) #3952
val_df = pd.DataFrame(val) #240
wiki_df = pd.DataFrame(wiki_dataset) #60613
test_df = pd.DataFrame(test) #600
